In [20]:
import pandas as pd
import multixrank
import numpy as np
import networkx as nx
import pickle

In [21]:
def load_multiplex(pkl_path: str) -> nx.Graph:
    print(f"Loading multiplex from {pkl_path} ...")
    with open(pkl_path, "rb") as f:
        G = pickle.load(f)
    print(f"  {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges\n")
    return G

In [1]:
# Quick inspection
import pickle, networkx as nx
H = pickle.load(open("multixrank_input_consolidated/gene_reference_graph.pkl", "rb"))
sample_edges = list(H.edges(data=True))[:3]
print(sample_edges)

[('SPCS1', 'TM2D2', {'cell_types': {'ppi_layer', 'A-LTMR.TAC3_layer', 'A-PEP.KIT_layer', 'C-PEP.ADORA2B_layer'}, 'inter_cell_types': set(), 'edge_type': 'ppi'}), ('SPCS1', 'AGPAT3', {'cell_types': {'ppi_layer', 'A-LTMR.TAC3_layer', 'A-PEP.KIT_layer'}, 'inter_cell_types': set(), 'edge_type': 'ppi'}), ('SPCS1', 'CAVIN1', {'cell_types': {'ppi_layer', 'A-LTMR.TAC3_layer', 'C-PEP.ADORA2B_layer'}, 'inter_cell_types': set(), 'edge_type': 'ppi'})]


In [1]:
import pickle
from collections import Counter

G = pickle.load(open("03_integrated/integrated_multiplex_output/integrated_multiplex.pkl", "rb"))

# Count edge types
edge_types = Counter()
for u, v, data in G.edges(data=True):
    edge_type = data.get("edge_type", "unknown")
    edge_types[edge_type] += 1

print("Edge types in multiplex:")
for et, count in sorted(edge_types.items(), key=lambda x: -x[1]):
    print(f"  {et:<30} {count:>15,}")

Edge types in multiplex:
  ppi                                 24,447,347
  ligand_receptor                     14,971,740


In [2]:
#!/usr/bin/env python3
"""Debug what edge types are actually in the multiplex"""
import pickle
from collections import defaultdict

multiplex_pkl = "03_integrated/integrated_multiplex_output/integrated_multiplex.pkl"

with open(multiplex_pkl, "rb") as f:
    G = pickle.load(f)

edge_type_counts = defaultdict(int)
for u, v, d in G.edges(data=True):
    et = d.get("edge_type", "MISSING")
    edge_type_counts[et] += 1

print("Edge types in multiplex:")
for et in sorted(edge_type_counts.keys()):
    print(f"  {et:<25} {edge_type_counts[et]:>12,}")

# Check if any edges have pathway info
pathway_count = 0
for u, v, d in G.edges(data=True):
    if "pathway" in d or "pathways" in d:
        pathway_count += 1

print(f"\nEdges with pathway metadata: {pathway_count:,}")

# Sample a few pathway edges
print("\nSample edges with 'pathway' metadata:")
count = 0
for u, v, d in G.edges(data=True):
    if "pathway" in d or "pathways" in d:
        print(f"  {d}")
        count += 1
        if count >= 5:
            break

Edge types in multiplex:
  ligand_receptor             14,971,740
  ppi                         24,447,347

Edges with pathway metadata: 0

Sample edges with 'pathway' metadata:


In [4]:
def split_layers(G, idx_of, n, refine_inter=False, lr_set=None, scored_lr_set=None, pw_set=None):
    rows = defaultdict(list); cols = defaultdict(list); data = defaultdict(list)
    for u, v, attr in G.edges(data=True):
        et = attr.get("edge_type", "ppi")
        w = 1.0
        i, j = idx_of[u], idx_of[v]

        target_layers = []
        if et in ("ppi", "both"):
            target_layers.append("ppi")
        if et in ("inter", "both"):
            if refine_inter:
                target_layers.append(refine_inter_edge_type(u, v, lr_set, scored_lr_set, pw_set))
            else:
                target_layers.append("inter")

        for layer in target_layers:
            rows[layer] += [i, j]; cols[layer] += [j, i]; data[layer] += [w, w]

    layers = {}
    for et in rows:
        A = csr_matrix((data[et], (rows[et], cols[et])), shape=(n, n))
        A.sum_duplicates(); layers[et] = A
    return layers


In [7]:
import pickle, networkx as nx
import os, json, pickle, argparse, time
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import networkx as nx
from scipy.sparse import csr_matrix, diags, identity, bmat
from scipy.sparse.linalg import eigsh
from sklearn.utils.extmath import randomized_svd
import matplotlib.pyplot as plt

with open("04_multixrank_prep/multixrank_hetero_input/gene_reference_graph.pkl", "rb") as f:
    G = pickle.load(f)
G = nx.relabel_nodes(G, {n: str(n).strip() for n in G.nodes()})
nodelist = sorted(G.nodes())
idx_of = {n: i for i, n in enumerate(nodelist)}

n = len(nodelist)


layers = split_layers(G, idx_of, n)

In [8]:
from scipy.sparse.csgraph import connected_components
import numpy as np

combined = sum(layers.values())  # ppi + inter, added together
n_comp, labels = connected_components(combined, directed=False)
print(f"Number of connected pieces: {n_comp}")

sizes = np.bincount(labels)
sizes_sorted = np.sort(sizes)[::-1]
print(f"Size of largest piece: {sizes_sorted[0]}")
print(f"Sizes of next 10 pieces: {sizes_sorted[1:11]}")
print(f"How many genes are in tiny pieces (<5 genes): {sizes[sizes < 5].sum()}")

Number of connected pieces: 1
Size of largest piece: 78890
Sizes of next 10 pieces: []
How many genes are in tiny pieces (<5 genes): 0


In [9]:
import pandas as pd
df = pd.read_csv("09_marchenko_pastur/multiplex_bridges.csv")
print(df["fiedler_mean"].describe())
print((df["fiedler_mean"] > 0).sum(), "positive vs", (df["fiedler_mean"] < 0).sum(), "negative")

count    7.889000e+04
mean     9.094035e-15
std      2.514373e-03
min     -8.908579e-04
25%     -4.263848e-05
50%     -4.197495e-05
75%     -4.056190e-05
max      1.521009e-01
Name: fiedler_mean, dtype: float64
38 positive vs 78852 negative


In [10]:
df = pd.read_csv("09_marchenko_pastur/multiplex_bridges.csv")
positive_genes = df.loc[df["fiedler_mean"] > 0, ["node_id", "cell_type", "gene"]]
print(positive_genes)

                            node_id             cell_type      gene
0                global_ppi::SEMA3G            global_ppi    SEMA3G
1            C-Thermo.RXFP1::SEMA3G        C-Thermo.RXFP1    SEMA3G
2        C-NP.MRGPRX1_GFRA2::SEMA3G    C-NP.MRGPRX1_GFRA2    SEMA3G
3                      ATF3::SEMA3G                  ATF3    SEMA3G
4               A-LTMR.TAC3::SEMA3G           A-LTMR.TAC3    SEMA3G
5      A-PEP.CHRNA7_SLC18A3::SEMA3G  A-PEP.CHRNA7_SLC18A3    SEMA3G
6              Ab-LTMR.ETV1::SEMA3G          Ab-LTMR.ETV1    SEMA3G
7         C-PEP.TAC1_CACNG5::SEMA3G     C-PEP.TAC1_CACNG5    SEMA3G
8              Ab-LTMR.LGI2::SEMA3G          Ab-LTMR.LGI2    SEMA3G
9            C-Thermo.TRPM8::SEMA3G        C-Thermo.TRPM8    SEMA3G
10             Ab-LTMR.NSG2::SEMA3G          Ab-LTMR.NSG2    SEMA3G
11            A-Propr.PCDH8::SEMA3G         A-Propr.PCDH8    SEMA3G
12                 C-NP.SST::SEMA3G              C-NP.SST    SEMA3G
13              C-LTMR.CDH9::SEMA3G           C-

In [11]:
from scipy.sparse.csgraph import connected_components
combined = sum(layers.values())
n_comp, labels = connected_components(combined, directed=False)
sizes = np.bincount(labels)
print(f"{n_comp} connected pieces; largest = {sizes.max()}; "
      f"sizes of all others: {sorted(sizes[sizes < sizes.max()], reverse=True)}")

1 connected pieces; largest = 78890; sizes of all others: []


In [12]:
from scipy.sparse.csgraph import connected_components

combined = sum(layers.values())
n_comp, labels = connected_components(combined, directed=False)
main_label = np.bincount(labels).argmax()
keep_mask = labels == main_label
print(f"Keeping {keep_mask.sum():,} / {n:,} nodes in largest connected piece "
      f"({n_comp} pieces total, dropping {(~keep_mask).sum():,} genes)")

layers = {name: A[keep_mask][:, keep_mask] for name, A in layers.items()}
nodelist = [nd for nd, k in zip(nodelist, keep_mask) if k]
n = len(nodelist)
idx_of = {nd: i for i, nd in enumerate(nodelist)}

Keeping 78,890 / 78,890 nodes in largest connected piece (1 pieces total, dropping 0 genes)


In [13]:
combined = sum(layers.values())
deg = np.asarray(combined.sum(axis=1)).ravel()

flagged_idx = [idx_of[nid] for nid in positive_genes["node_id"]]  # your 38 positive genes
print("Degree of the 38 flagged genes:")
print(pd.Series(deg[flagged_idx]).describe())

print("\nDegree of everyone else, for comparison:")
print(pd.Series(deg).describe())

Degree of the 38 flagged genes:
count    38.000000
mean     15.710526
std      25.087854
min       1.000000
25%       1.000000
50%       1.000000
75%      19.750000
max      85.000000
dtype: float64

Degree of everyone else, for comparison:
count    78890.000000
mean        15.921764
std         30.060886
min          1.000000
25%          3.000000
50%          7.000000
75%         16.000000
max       1532.000000
dtype: float64


In [16]:
df = pd.read_csv("09_marchenko_pastur/multiplex_bridges.csv")
print(df["n_celltypes_present"].describe())
print(df[df["is_multiplex_bridge"]]["n_celltypes_present"].value_counts().sort_index())

count    1183.000000
mean        3.715976
std         4.006822
min         1.000000
25%         1.000000
50%         2.000000
75%         5.000000
max        22.000000
Name: n_celltypes_present, dtype: float64
n_celltypes_present
2     187
3     121
4      83
5      62
6      50
7      31
8      23
9      17
10     18
11     17
12     15
13     13
14      7
15     10
16     13
17      5
18      7
19      9
20      3
22      1
Name: count, dtype: int64


In [17]:
rate = (df[df.is_multiplex_bridge].groupby("n_celltypes_present").size()
        / df.groupby("n_celltypes_present").size())
print(rate.sort_index())

n_celltypes_present
1          NaN
2     0.857798
3     1.000000
4     1.000000
5     1.000000
6     0.980392
7     1.000000
8     1.000000
9     1.000000
10    1.000000
11    1.000000
12    1.000000
13    1.000000
14    1.000000
15    1.000000
16    1.000000
17    1.000000
18    1.000000
19    1.000000
20    1.000000
22    1.000000
dtype: float64


In [18]:
import pickle
from collections import defaultdict

with open("06_disease_module/disease_module_output/module_graph.pkl", "rb") as f:
    G = pickle.load(f)

def ct_of(n): return n.rsplit("::",1)[0] if "::" in n else "unknown"
def gene_of(n): return n.rsplit("::",1)[-1] if "::" in n else n

gene_cts = defaultdict(set)
for n in G.nodes():
    if ct_of(n).lower() != "global_ppi":
        gene_cts[gene_of(n)].add(ct_of(n))

gp_edges = [(u,v) for u,v,d in G.edges(data=True)
            if ct_of(u).lower()=="global_ppi" and ct_of(v).lower()=="global_ppi"]
n_overlap = sum(1 for u,v in gp_edges
                if gene_cts.get(gene_of(u),set()) & gene_cts.get(gene_of(v),set()))
n_either_present = sum(1 for u,v in gp_edges
                        if gene_cts.get(gene_of(u)) or gene_cts.get(gene_of(v)))
print(f"{len(gp_edges)} global_ppi edges, {n_overlap} with same-celltype overlap, "
      f"{n_either_present} where at least one gene has ANY celltype presence")

318 global_ppi edges, 248 with same-celltype overlap, 315 where at least one gene has ANY celltype presence


In [19]:
import networkx as nx
G2 = G.subgraph([n for n in G.nodes() if not n.startswith("global_ppi::")]).copy()
iso = [n for n in G2.nodes() if G2.degree(n) == 0]
G3 = G2.subgraph([n for n in G2.nodes() if n not in set(iso)]).copy()
print("true connected components:", nx.number_connected_components(G3))
for i, cc in enumerate(sorted(nx.connected_components(G3), key=len, reverse=True)):
    print(f"  component {i}: {len(cc)} nodes")

true connected components: 51
  component 0: 4280 nodes
  component 1: 5 nodes
  component 2: 4 nodes
  component 3: 4 nodes
  component 4: 4 nodes
  component 5: 3 nodes
  component 6: 3 nodes
  component 7: 3 nodes
  component 8: 3 nodes
  component 9: 3 nodes
  component 10: 3 nodes
  component 11: 3 nodes
  component 12: 2 nodes
  component 13: 2 nodes
  component 14: 2 nodes
  component 15: 2 nodes
  component 16: 2 nodes
  component 17: 2 nodes
  component 18: 2 nodes
  component 19: 2 nodes
  component 20: 2 nodes
  component 21: 2 nodes
  component 22: 2 nodes
  component 23: 2 nodes
  component 24: 2 nodes
  component 25: 2 nodes
  component 26: 2 nodes
  component 27: 2 nodes
  component 28: 2 nodes
  component 29: 2 nodes
  component 30: 2 nodes
  component 31: 2 nodes
  component 32: 2 nodes
  component 33: 2 nodes
  component 34: 2 nodes
  component 35: 2 nodes
  component 36: 2 nodes
  component 37: 2 nodes
  component 38: 2 nodes
  component 39: 2 nodes
  component 40: 2

In [20]:
import pandas as pd
df = pd.read_csv("09_marchenko_pastur/multiplex_bridges.csv")
testable = df[~df["insufficient_data"]]
print(testable["frac_neg_layers"].describe())
print(testable["frac_neg_layers"].value_counts(bins=10).sort_index())

count    506.000000
mean       0.054423
std        0.094241
min        0.000000
25%        0.000000
50%        0.000000
75%        0.089015
max        0.500000
Name: frac_neg_layers, dtype: float64
(-0.0015, 0.05]    342
(0.05, 0.1]         50
(0.1, 0.15]         30
(0.15, 0.2]         49
(0.2, 0.25]         15
(0.25, 0.3]          3
(0.3, 0.35]         14
(0.35, 0.4]          0
(0.4, 0.45]          0
(0.45, 0.5]          3
Name: count, dtype: int64


In [21]:
import pickle
from collections import defaultdict

with open("06_disease_module/disease_module_output/module_graph.pkl", "rb") as f:
    G = pickle.load(f)

def ct_of(n): return n.rsplit("::",1)[0] if "::" in n else "unknown"
def gene_of(n): return n.rsplit("::",1)[-1] if "::" in n else n

gene_cts = defaultdict(set)
for n in G.nodes():
    if ct_of(n).lower() != "global_ppi":
        gene_cts[gene_of(n)].add(ct_of(n))

gp_edges = [(u,v) for u,v,d in G.edges(data=True)
            if ct_of(u).lower()=="global_ppi" and ct_of(v).lower()=="global_ppi"]

exists, missing = 0, 0
for u, v in gp_edges:
    ga, gb = gene_of(u), gene_of(v)
    for ct in gene_cts.get(ga,set()) & gene_cts.get(gb,set()):
        if G.has_edge(f"{ct}::{ga}", f"{ct}::{gb}"):
            exists += 1
        else:
            missing += 1
print("overlap pairs where celltype edge already exists:", exists)
print("overlap pairs where it's genuinely missing (should project):", missing)

overlap pairs where celltype edge already exists: 1177
overlap pairs where it's genuinely missing (should project): 0


In [22]:
import numpy as np
fiedler = np.load("09_marchenko_pastur/fiedler_vector.npy")
print("total:", len(fiedler), "negative:", (fiedler<0).sum(), "positive:", (fiedler>0).sum())

total: 4280 negative: 4045 positive: 235


In [3]:
import pandas as pd
df = pd.read_csv("09_marchenko_pastur/permutation_test/bridge_permutation_results.csv")
top = df[df["testable"]].sort_values("perm_pvalue").head(20)
print(top[["gene","n_celltypes_present","real_frac_minority","perm_pvalue","perm_qvalue_bh"]])

       gene  n_celltypes_present  real_frac_minority  perm_pvalue  \
0   ADCYAP1                   20            0.250000     0.000100   
1    PRKAA2                    5            0.200000     0.993801   
2   BLOC1S6                    3            0.333333     0.994201   
3      NOS1                   19            0.052632     0.994301   
4     CTXN1                    8            0.125000     0.994301   
5    CACNB3                    6            0.166667     0.994401   
6     PRDX5                    5            0.200000     0.994501   
7      GNAS                    6            0.166667     0.994501   
8      RFFL                    6            0.166667     0.994501   
9    PLXDC2                    7            0.142857     0.994601   
10    PCSK6                   17            0.058824     0.994701   
11      PXN                    6            0.166667     0.994801   
12    PRKG1                   14            0.071429     0.994801   
15   KCNIP3                   15  

In [4]:
#!/usr/bin/env python3
"""
Quick diagnostic for step 3c/11c OOMs: count how many total clusters your
step 3b ensemble actually produces, and estimate the memory a dense
O(n_clusters^2) pairwise comparison inside weave() would need. Run this
directly on a login node -- it's seconds, not a SLURM job -- BEFORE
requesting more memory for step 3c.

Usage:
    python check_ensemble_size.py --ensemble /path/to/ensemble.pkl
"""
import argparse
import pickle

import numpy as np


with open("11_hidef_hetero/ensemble.pkl", "rb") as f:
    data = pickle.load(f)

partitions = data["partitions"]
n_nodes = len(data["node_names"])
print(f"n_nodes = {n_nodes:,}, n_partitions = {len(partitions)}\n")

n_clusters_total = 0
size_hist = {"1": 0, "2-4": 0, "5-9": 0, "10-99": 0, "100+": 0}
per_partition = []

for p in partitions:
    membership = np.asarray(p["membership"])
    ids, counts = np.unique(membership, return_counts=True)
    per_partition.append((p["gamma"], p["omega"], len(ids), int(counts.min()), int(counts.max())))
    n_clusters_total += len(ids)
    for c in counts:
        if c == 1:
            size_hist["1"] += 1
        elif c < 5:
            size_hist["2-4"] += 1
        elif c < 10:
            size_hist["5-9"] += 1
        elif c < 100:
            size_hist["10-99"] += 1
        else:
            size_hist["100+"] += 1

print(f"Total clusters across all partitions: {n_clusters_total:,}")
print(f"Size distribution (cluster count by size bucket): {size_hist}")
est_gb = 8 * n_clusters_total ** 2 / 1e9
print(f"\nEstimated memory if weave() does dense O(n_clusters^2) float64 pairwise "
      f"comparison: ~{est_gb:.1f} GB")
print("(this is the most likely OOM culprit, not the input cluster strings themselves --")
print(" those would only be ~{:.1f} GB even as plain Python strings)\n"
      .format(sum(len(str(i)) for i in range(n_clusters_total)) / 1e9 if False else
              n_clusters_total * n_nodes / 1e9))

print("gamma, omega, n_communities, min_size, max_size (per partition):")
for row in per_partition:
    print(f"  gamma={row[0]:.4g} omega={row[1]:.4g} -> {row[2]:,} communities "
          f"(size {row[3]}-{row[4]})")

n_singletons = size_hist["1"] + size_hist["2-4"]
if n_singletons > 0.3 * n_clusters_total:
    print(f"\n⚠ {n_singletons:,} of {n_clusters_total:,} clusters ({100*n_singletons/n_clusters_total:.0f}%) "
          f"have <5 members -- these are almost certainly the biggest driver of cluster count "
          f"blowup at high resolution and contribute little persistence signal. Filtering them "
          f"out before weave() (e.g. drop clusters with <5 members) could shrink n_clusters^2 "
          f"dramatically with minimal loss of real signal.")

n_nodes = 140,139, n_partitions = 30

Total clusters across all partitions: 271,653
Size distribution (cluster count by size bucket): {'1': 0, '2-4': 108210, '5-9': 90573, '10-99': 71671, '100+': 1199}

Estimated memory if weave() does dense O(n_clusters^2) float64 pairwise comparison: ~590.4 GB
(this is the most likely OOM culprit, not the input cluster strings themselves --
 those would only be ~38.1 GB even as plain Python strings)

gamma, omega, n_communities, min_size, max_size (per partition):
  gamma=0.01 omega=1 -> 9,030 communities (size 2-17051)
  gamma=0.01269 omega=1 -> 9,030 communities (size 2-17051)
  gamma=0.0161 omega=1 -> 9,030 communities (size 2-17051)
  gamma=0.02043 omega=1 -> 9,030 communities (size 2-17051)
  gamma=0.02593 omega=1 -> 9,032 communities (size 2-17051)
  gamma=0.0329 omega=1 -> 9,032 communities (size 2-17051)
  gamma=0.04175 omega=1 -> 9,032 communities (size 2-17051)
  gamma=0.05298 omega=1 -> 9,033 communities (size 2-17051)
  gamma=0.06723 omeg

In [7]:
import pandas as pd
df = pd.read_csv("08_spectral_analysis/module_with_ppi/spectral_clusters.csv")

# 1. Does cluster assignment just recover cell type?
ct = pd.crosstab(df.cluster, df.cell_type, normalize='index')
print(ct.round(2))
print("\nmax cell-type share per cluster:", ct.max(axis=1).values)

# 2. Cluster size balance — k-means with silhouette 0.9 sometimes hides
#    one giant blob + several tiny outlier clusters
print(df.cluster.value_counts())

# 3. Are clusters just separating by degree/hub-ness (a known SoS artifact)?
print(df.groupby('cluster')[['deg_ppi','deg_inter']].mean())

# 4. Role distribution per cluster — genuine multiplex integration should
#    put some "bridge"/"multiplex_hub" nodes in every cluster, not
#    concentrated in one
print(pd.crosstab(df.cluster, df.role, normalize='index').round(2))

cell_type  A-LTMR.TAC3  A-PEP.CHRNA7_SLC18A3  A-PEP.KIT  A-PEP.NTRK3_S100A16  \
cluster                                                                        
0                 0.03                  0.05       0.04                 0.05   
1                 0.00                  0.00       0.00                 0.00   
2                 0.00                  0.00       0.00                 0.00   
3                 0.00                  0.00       0.00                 0.00   
4                 0.00                  0.00       0.00                 0.00   
5                 0.00                  0.00       0.00                 0.00   
6                 0.00                  0.00       0.00                 0.00   
7                 0.00                  0.00       0.00                 0.00   
8                 0.00                  0.00       0.00                 0.00   

cell_type  A-PEP.SCGN_ADRA2C  A-Propr.EPHA3  A-Propr.HAPLN4  A-Propr.PCDH8  \
cluster                                  

In [8]:
import pandas as pd
df = pd.read_csv("08_spectral_analysis/module_no_ppi/spectral_clusters.csv")

# 1. Does cluster assignment just recover cell type?
ct = pd.crosstab(df.cluster, df.cell_type, normalize='index')
print(ct.round(2))
print("\nmax cell-type share per cluster:", ct.max(axis=1).values)

# 2. Cluster size balance — k-means with silhouette 0.9 sometimes hides
#    one giant blob + several tiny outlier clusters
print(df.cluster.value_counts())

# 3. Are clusters just separating by degree/hub-ness (a known SoS artifact)?
print(df.groupby('cluster')[['deg_ppi','deg_inter']].mean())

# 4. Role distribution per cluster — genuine multiplex integration should
#    put some "bridge"/"multiplex_hub" nodes in every cluster, not
#    concentrated in one
print(pd.crosstab(df.cluster, df.role, normalize='index').round(2))

cell_type  A-LTMR.TAC3  A-PEP.CHRNA7_SLC18A3  A-PEP.KIT  A-PEP.NTRK3_S100A16  \
cluster                                                                        
0                 0.04                  0.05       0.04                 0.05   
1                 0.00                  0.00       0.00                 0.00   
2                 0.00                  0.00       0.00                 0.00   
3                 0.00                  0.00       0.00                 0.00   

cell_type  A-PEP.SCGN_ADRA2C  A-Propr.EPHA3  A-Propr.HAPLN4  A-Propr.PCDH8  \
cluster                                                                      
0                       0.05           0.04            0.05           0.04   
1                       0.00           0.00            0.00           0.00   
2                       0.00           0.00            0.00           0.00   
3                       0.00           1.00            0.00           0.00   

cell_type  ATF3  Ab-LTMR.CCKAR  ...  Ab-LTMR.NSG2 

In [1]:
import pandas as pd
from sklearn.metrics import adjusted_rand_score

for path, label in [
    ("08_spectral_analysis/module_with_ppi/spectral_clusters.csv", "with_ppi"),
    ("08_spectral_analysis/module_no_ppi/spectral_clusters.csv", "no_ppi"),
]:
    df = pd.read_csv(path)
    ari = adjusted_rand_score(df["cell_type"], df["cluster"])
    purity = df.groupby("cluster")["cell_type"].agg(lambda s: s.value_counts().iloc[0] / len(s)).mean()
    n_celltypes = df["cell_type"].nunique()
    print(f"{label}: {n_celltypes} cell types, cluster-vs-celltype ARI={ari:.4f}, purity={purity:.4f}")

with_ppi: 23 cell types, cluster-vs-celltype ARI=-0.0000, purity=0.8945
no_ppi: 22 cell types, cluster-vs-celltype ARI=0.0000, purity=0.7634


In [3]:
#!/usr/bin/env python3
"""
check_hidef_communities.py

Standalone sanity check for step 11c's output
(heterogeneous_hidef_communities.pkl) and, optionally, step 11d's
cross-check report (infomap_crosscheck.json).

WHAT THIS CHECKS
----------------
1. How many entries are in the communities pickle, and what their size
   distribution looks like. If len(communities) is suspiciously close to
   your total node count (~140k), and most entries have size == 1, that
   means step 11c's extract_communities() is emitting one "community" per
   leaf/terminal node rather than per persistent hierarchy community --
   which would make a purity-against-Infomap comparison trivially 1.0 for
   almost every entry (a single node is always "pure").

2. Cross-references against infomap_crosscheck.json's `per_community`
   section (if you pass --report), so you can see directly whether the
   n=140139 in your last report lines up with genuinely small/degenerate
   communities.

3. Prints a handful of concrete example entries so you can eyeball
   whether `members`/`size`/`persistence` look like real communities
   (dozens-thousands of genes across cell types) or single terminal nodes.

USAGE
-----
    python check_hidef_communities.py \
        --communities 11_hidef_results/heterogeneous_hidef_communities.pkl \
        --report 11_hidef_results/infomap_crosscheck.json
"""
import argparse
import json
import pickle
from collections import Counter
from pathlib import Path

import numpy as np


def load_communities(path: Path):
    with open(path, "rb") as f:
        communities = pickle.load(f)
    if not isinstance(communities, dict):
        raise TypeError(
            f"Expected a dict of {{community_id: community_dict}}, got {type(communities)}"
        )
    return communities


def summarize_sizes(communities: dict):
    sizes = []
    for cid, c in communities.items():
        if isinstance(c, dict) and "size" in c:
            sizes.append(c["size"])
        elif isinstance(c, dict) and "members" in c:
            sizes.append(len(c["members"]))
        else:
            # Unexpected shape -- try to infer length directly
            try:
                sizes.append(len(c))
            except TypeError:
                sizes.append(None)
    return sizes


def print_size_distribution(sizes, n_nodes_hint=None):
    sizes_clean = [s for s in sizes if s is not None]
    n_none = len(sizes) - len(sizes_clean)
    arr = np.array(sizes_clean)

    print(f"Total entries in communities dict: {len(sizes):,}")
    if n_none:
        print(f"  ⚠ {n_none} entries had no interpretable size (unexpected shape)")

    if arr.size == 0:
        print("  ⚠ No usable size values found -- can't summarize further.")
        return

    print(f"Size stats: min={arr.min()}  max={arr.max()}  mean={arr.mean():.2f}  "
          f"median={np.median(arr):.1f}")

    n_singletons = int((arr == 1).sum())
    frac_singletons = n_singletons / arr.size
    print(f"Singleton communities (size == 1): {n_singletons:,} / {arr.size:,} "
          f"({frac_singletons:.1%})")

    if n_nodes_hint is not None:
        ratio = len(sizes) / n_nodes_hint
        print(f"\nEntries-to-total-nodes ratio: {len(sizes):,} / {n_nodes_hint:,} "
              f"= {ratio:.3f}")
        if ratio > 0.5:
            print("  ⚠ This is suspiciously close to 1 node-per-entry. If most "
                  "entries also have size == 1, extract_communities() in step "
                  "11c is very likely emitting terminal/leaf nodes rather than "
                  "persistent hierarchy communities, and any purity computed "
                  "against this dict will be inflated toward 1.0.")

    # Small histogram of size buckets, since a handful of numbers won't show
    # bimodality (e.g. many singletons + a few real communities) as clearly.
    buckets = [1, 2, 5, 10, 50, 100, 500, 1000, 5000, np.inf]
    labels = ["1", "2-4", "5-9", "10-49", "50-99", "100-499",
              "500-999", "1000-4999", "5000+"]
    counts = Counter()
    for s in arr:
        for i in range(len(buckets) - 1):
            if buckets[i] <= s < buckets[i + 1] or (i == 0 and s == 1):
                counts[labels[min(i, len(labels) - 1)]] += 1
                break
    print("\nSize histogram:")
    for label in labels:
        if counts[label]:
            print(f"  {label:>12}: {counts[label]:,}")


def print_examples(communities: dict, n=5):
    print(f"\n--- {n} example entries ---")
    for i, (cid, c) in enumerate(communities.items()):
        if i >= n:
            break
        if isinstance(c, dict):
            members = c.get("members", [])
            size = c.get("size", len(members) if members else "?")
            persistence = c.get("persistence", "n/a")
            n_cell_types = c.get("n_cell_types_spanned", "n/a")
            preview = members[:3] if isinstance(members, list) else members
            print(f"  id={cid!r}  size={size}  persistence={persistence}  "
                  f"n_cell_types_spanned={n_cell_types}  members_preview={preview}")
        else:
            print(f"  id={cid!r}  raw_value={c!r}")


def cross_check_report(report_path: Path, communities: dict):
    with open(report_path) as f:
        report = json.load(f)

    per_community = report.get("per_community", {})
    print(f"\n--- Cross-check against {report_path.name} ---")
    print(f"Report's per_community entries: {len(per_community):,}")
    print(f"Communities pickle entries:      {len(communities):,}")
    if len(per_community) != len(communities):
        print("  ⚠ Counts differ -- some communities pickle entries had no "
              "members present in the supra-graph (dropped silently by "
              "community_purity()), or vice versa.")
    else:
        print("  ✓ Counts match.")

    mean_purity = report.get("mean_community_purity")
    median_purity = report.get("median_community_purity")
    print(f"Report mean purity: {mean_purity}   median purity: {median_purity}")

    if mean_purity is not None and mean_purity > 0.999:
        # Check whether that's being driven by singleton communities
        sizes = summarize_sizes(communities)
        arr = np.array([s for s in sizes if s is not None])
        if arr.size and (arr == 1).mean() > 0.5:
            print("  ⚠ Purity is ~1.0 AND most communities are singletons -- "
                  "the purity statistic in this report is very likely "
                  "meaningless as a cross-validation of step 11c's hierarchy "
                  "against Infomap. Fix extract_communities() in step 11c to "
                  "only emit hierarchy nodes with an 'index' attribute "
                  "pointing at multi-node clusters (see idx = attrs.get(\"index\") "
                  "handling), and re-run 11c before trusting this report.")


communities = load_communities(Path("11_hidef_results/heterogeneous_hidef_communities.pkl"))

sizes = summarize_sizes(communities)
print_size_distribution(sizes, n_nodes_hint=140139)
print_examples(communities, n=5)

cross_check_report(Path("11_hidef_results/infomap_crosscheck.json"), communities)



Total entries in communities dict: 140,139
Size stats: min=1  max=1  mean=1.00  median=1.0
Singleton communities (size == 1): 140,139 / 140,139 (100.0%)

Entries-to-total-nodes ratio: 140,139 / 140,139 = 1.000
  ⚠ This is suspiciously close to 1 node-per-entry. If most entries also have size == 1, extract_communities() in step 11c is very likely emitting terminal/leaf nodes rather than persistent hierarchy communities, and any purity computed against this dict will be inflated toward 1.0.

Size histogram:
             1: 140,139

--- 5 example entries ---
  id='0'  size=1  persistence=None  n_cell_types_spanned=1  members_preview=['A-LTMR.TAC3::A-LTMR.TAC3::A4GALT']
  id='1'  size=1  persistence=None  n_cell_types_spanned=1  members_preview=['A-LTMR.TAC3::A-LTMR.TAC3::AAAS']
  id='2'  size=1  persistence=None  n_cell_types_spanned=1  members_preview=['A-LTMR.TAC3::A-LTMR.TAC3::AAGAB']
  id='3'  size=1  persistence=None  n_cell_types_spanned=1  members_preview=['A-LTMR.TAC3::A-LTMR.TAC3

In [11]:
import json
import pickle
from collections import defaultdict
from pathlib import Path

import numpy as np
import networkx as nx

# ---- set these ----
ENSEMBLE_PATH = "11_hidef_hetero/ensemble.pkl"
GML_PATH = "11_hidef_results/heterogeneous_hidef_hierarchy.gml"
OUTPUT_DIR = "11_hidef_results_fixed"
MIN_CLUSTER_SIZE = 10   # must match your original step 11c run
DEDUP = True           # must match your original step 11c run (--no-dedup was NOT used)
MAX_SIZE = 4680

In [12]:
def destringize(s):
    """Undo write_gml(..., stringizer=str): try int, then float, then None,
    else leave as string."""
    if s is None:
        return None
    if isinstance(s, (int, float)):
        return s
    if s == "None":
        return None
    try:
        return int(s)
    except (TypeError, ValueError):
        pass
    try:
        return float(s)
    except (TypeError, ValueError):
        pass
    return s


import re

NODE_START = re.compile(r'^\s*node\s*\[\s*$')
EDGE_START = re.compile(r'^\s*edge\s*\[\s*$')
KV_STRING = re.compile(r'^\s*(\w+)\s+"(.*)"\s*$')
KV_BARE = re.compile(r'^\s*(\w+)\s+(\S+)\s*$')

WANTED_KEYS = ("id", "label", "index", "persistence", "level")


def parse_gml_nodes_tolerant(path, wanted_keys=WANTED_KEYS):
    """Streams a GML file line-by-line and extracts only node-level
    attributes we care about, skipping malformed lines/blocks instead of
    raising. Ignores edges entirely (we don't need them for community
    recovery). Handles a corrupted/truncated file gracefully -- reports
    how many node blocks and how many unparseable lines it hit."""
    nodes = []
    n_malformed = 0
    n_node_blocks_seen = 0
    in_node, in_edge = False, False
    depth = 0
    current = {}

    with open(path, "r", errors="replace") as f:
        for line in f:
            if not in_node and not in_edge:
                if NODE_START.match(line):
                    in_node, depth, current = True, 1, {}
                    n_node_blocks_seen += 1
                elif EDGE_START.match(line):
                    in_edge, depth = True, 1
                continue

            stripped = line.strip()
            if stripped.endswith("["):
                depth += 1
                continue
            if stripped == "]":
                depth -= 1
                if depth <= 0:
                    if in_node:
                        nodes.append(current)
                    in_node, in_edge = False, False
                continue

            if in_node and depth == 1:
                m = KV_STRING.match(line)
                if m:
                    key, val = m.groups()
                    if key in wanted_keys:
                        current[key] = val
                    continue
                m = KV_BARE.match(line)
                if m:
                    key, val = m.groups()
                    if key in wanted_keys:
                        current[key] = val
                    continue
                n_malformed += 1
            # inside an edge block or nested dict we don't care about -- skip

    print(f"Node blocks opened: {n_node_blocks_seen:,}")
    print(f"Node blocks fully parsed: {len(nodes):,}")
    if n_node_blocks_seen != len(nodes):
        print(f"  ⚠ {n_node_blocks_seen - len(nodes):,} node block(s) never closed cleanly -- "
              f"likely the point where the file gets corrupted/truncated. Everything "
              f"before that point should still be usable.")
    print(f"Unparseable attribute lines skipped: {n_malformed:,}")
    return nodes


def inspect_gml(path, n=8):
    print(f"Loading {path} (tolerant parser) ...")
    nodes = parse_gml_nodes_tolerant(path)

    with_index = [nd for nd in nodes if "index" in nd]
    without_index = [nd for nd in nodes if "index" not in nd]

    print(f"\nNodes with an 'index' (candidate internal clusters): {len(with_index):,}")
    print(f"Nodes without one (candidate terminal leaves): {len(without_index):,}\n")

    print(f"--- {min(n, len(with_index))} example 'index' nodes ---")
    for nd in with_index[:n]:
        print(f"  {nd}")

    print(f"\n--- {min(n, len(without_index))} example leaf nodes ---")
    for nd in without_index[:n]:
        print(f"  {nd}")

    return nodes

def load_ensemble(path):
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data["partitions"], data["node_names"], data["cell_type"], data["gene"]


def partitions_to_cluster_strings(partitions, n_nodes, min_cluster_size=1, dedup=True):
    raw_bits_by_hash = {}
    n_raw = 0
    n_dropped_small = 0
    for p in partitions:
        membership = np.asarray(p["membership"], dtype=int)
        if len(membership) != n_nodes:
            raise ValueError(f"partition (gamma={p['gamma']}, omega={p['omega']}, "
                              f"restart={p['restart']}) has {len(membership)} entries, "
                              f"expected {n_nodes} -- ensemble and node_names are out of sync")
        ids, counts = np.unique(membership, return_counts=True)
        for comm_id, size in zip(ids, counts):
            n_raw += 1
            if size < min_cluster_size:
                n_dropped_small += 1
                continue
            bits = (membership == comm_id).astype(np.uint8)
            key = bits.tobytes() if dedup else (bits.tobytes(), n_raw)
            entry = raw_bits_by_hash.get(key)
            if entry is None:
                raw_bits_by_hash[key] = {"bits": bits, "n_resolutions_seen": 1}
            else:
                entry["n_resolutions_seen"] += 1

    unique_clusters = list(raw_bits_by_hash.values())
    P_bits = [c["bits"] for c in unique_clusters]
    prevalence = [c["n_resolutions_seen"] for c in unique_clusters]
    print(f"Rebuilt {len(P_bits):,} unique clusters from {len(partitions)} partitions "
          f"(min_cluster_size={min_cluster_size}, dedup={dedup}) -- "
          f"must match your original step 11c settings or the GML 'index' values won't line up.")
    return P_bits, prevalence

In [13]:
G = inspect_gml(GML_PATH)

Loading 11_hidef_results/heterogeneous_hidef_hierarchy.gml (tolerant parser) ...
Node blocks opened: 143,035
Node blocks fully parsed: 143,035
Unparseable attribute lines skipped: 0

Nodes with an 'index' (candidate internal clusters): 2,896
Nodes without one (candidate terminal leaves): 140,139

--- 8 example 'index' nodes ---
  {'id': '0', 'label': '(2895,0)', 'index': '0', 'level': '0'}
  {'id': '1', 'label': '(2894,0)', 'index': '1', 'level': '1'}
  {'id': '2', 'label': '(2893,0)', 'index': '2', 'level': '2'}
  {'id': '3', 'label': '(2892,0)', 'index': '3', 'level': '3'}
  {'id': '4', 'label': '(2891,0)', 'index': '4', 'level': '4'}
  {'id': '5', 'label': '(2890,0)', 'index': '5', 'level': '5'}
  {'id': '6', 'label': '(2889,0)', 'index': '6', 'level': '6'}
  {'id': '7', 'label': '(2888,0)', 'index': '7', 'level': '7'}

--- 8 example leaf nodes ---
  {'id': '2896', 'label': '0'}
  {'id': '2897', 'label': '1'}
  {'id': '2898', 'label': '2'}
  {'id': '2899', 'label': '3'}
  {'id': '29

In [14]:
partitions, node_names, cell_types, genes = load_ensemble(ENSEMBLE_PATH)
print(f"Loaded ensemble: {len(partitions)} partitions, {len(node_names)} nodes\n")

P_bits, prevalence = partitions_to_cluster_strings(
    partitions, len(node_names), min_cluster_size=MIN_CLUSTER_SIZE, dedup=DEDUP,
)

communities = {}
n_internal, n_leaf, n_out_of_range, n_empty = 0, 0, 0, 0

for nd in G:   # G is the tolerant parser's list of node dicts, not a networkx graph
    idx = destringize(nd.get("index"))
    persistence = destringize(nd.get("persistence"))
    node_id = nd.get("id", nd.get("label"))

    if idx is not None:
        n_internal += 1
        if not isinstance(idx, int) or idx < 0 or idx >= len(P_bits):
            n_out_of_range += 1
            continue
        bits = P_bits[idx]
        members = [node_names[j] for j in np.nonzero(bits)[0]]
    else:
        n_leaf += 1
        leaf_idx = destringize(nd.get("label", node_id))
        if not isinstance(leaf_idx, int) or leaf_idx < 0 or leaf_idx >= len(node_names):
            continue
        members = [node_names[leaf_idx]]

    if not members:
        n_empty += 1
        continue
    if len(members) > MAX_SIZE:
        continue

    by_ct = defaultdict(list)
    for m in members:
        ct, gene = m.split("::", 1)
        by_ct[ct].append(gene)

    communities[str(node_id)] = {
        "members": members,
        "size": len(members),
        "persistence": persistence,
        "n_cell_types_spanned": len(by_ct),
        "members_by_cell_type": dict(by_ct),
    }

print(f"Nodes with 'index' (internal clusters): {n_internal:,} ({n_out_of_range:,} out-of-range)")
print(f"Nodes without one (terminal leaves): {n_leaf:,}")
print(f"Dropped for empty membership: {n_empty:,}")
print(f"\nRecovered {len(communities):,} communities (<= {MAX_SIZE} members)\n")

sizes = [c["size"] for c in communities.values()]
if sizes:
    n_singletons = sum(1 for s in sizes if s == 1)
    print(f"Size stats: min={min(sizes)} max={max(sizes)} mean={np.mean(sizes):.1f}")
    print(f"Singletons: {n_singletons:,}/{len(sizes):,} ({n_singletons/len(sizes):.1%}) "
          f"-- should be well below the 100% you got before, if this worked.")

Loaded ensemble: 30 partitions, 140139 nodes

Rebuilt 2,896 unique clusters from 30 partitions (min_cluster_size=10, dedup=True) -- must match your original step 11c settings or the GML 'index' values won't line up.
Nodes with 'index' (internal clusters): 2,896 (0 out-of-range)
Nodes without one (terminal leaves): 140,139
Dropped for empty membership: 0

Recovered 143,017 communities (<= 4680 members)

Size stats: min=1 max=4678 mean=3.3
Singletons: 140,139/143,017 (98.0%) -- should be well below the 100% you got before, if this worked.


In [15]:
out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

with open(out_dir / "heterogeneous_hidef_communities_FIXED.pkl", "wb") as f:
    pickle.dump(communities, f)

summary = {
    cid: {
        "size": c["size"], "persistence": c["persistence"],
        "n_cell_types_spanned": c["n_cell_types_spanned"],
        "cell_types": sorted(c["members_by_cell_type"].keys()),
    }
    for cid, c in communities.items()
}
with open(out_dir / "heterogeneous_hidef_communities_FIXED_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"✓ Saved {len(communities):,} communities -> {out_dir}")

✓ Saved 143,017 communities -> 11_hidef_results_fixed


In [16]:
import json
report = json.load(open("11_hidef_results_fixed/infomap_crosscheck_FIXED.json"))
pc = report["per_community"]
low_purity = sorted(pc.items(), key=lambda x: x[1]["purity"])[:15]
for cid, d in low_purity:
    print(cid, d)

2538 {'purity': 0.1425891181988743, 'top_infomap_module': 65, 'size': 533}
2424 {'purity': 0.15475918740013694, 'top_infomap_module': 13, 'size': 4381}
2426 {'purity': 0.15504230505373887, 'top_infomap_module': 13, 'size': 4373}
2429 {'purity': 0.16782178217821783, 'top_infomap_module': 13, 'size': 4040}
2436 {'purity': 0.168530947054437, 'top_infomap_module': 13, 'size': 4023}
2874 {'purity': 0.1791044776119403, 'top_infomap_module': 15, 'size': 67}
2471 {'purity': 0.18373275236020334, 'top_infomap_module': 27, 'size': 1377}
2537 {'purity': 0.18401486988847585, 'top_infomap_module': 33, 'size': 538}
2489 {'purity': 0.19182948490230906, 'top_infomap_module': 48, 'size': 563}
2631 {'purity': 0.19384615384615383, 'top_infomap_module': 1, 'size': 325}
2801 {'purity': 0.19413919413919414, 'top_infomap_module': 1, 'size': 273}
2583 {'purity': 0.2054794520547945, 'top_infomap_module': 84, 'size': 292}
2442 {'purity': 0.20893682588597842, 'top_infomap_module': 13, 'size': 3245}
2702 {'purity'

In [17]:
import pickle
from collections import Counter

import numpy as np
import pandas as pd

# ---- set this ----
COMMUNITIES_PATH = "11_hidef_results_fixed/heterogeneous_hidef_communities_FIXED.pkl"
MIN_SIZE = 2  # excludes trivial single-node leaf entries; set to 1 to include everything

with open(COMMUNITIES_PATH, "rb") as f:
    communities = pickle.load(f)

real = {cid: c for cid, c in communities.items() if c["size"] >= MIN_SIZE}
print(f"Total entries in pickle: {len(communities):,}")
print(f"Entries with size >= {MIN_SIZE}: {len(real):,}\n")

# ---- 1. counts / size / persistence / cell-type-span overview ----
df = pd.DataFrame([
    {
        "community_id": cid,
        "size": c["size"],
        "persistence": c.get("persistence"),
        "n_cell_types_spanned": c.get("n_cell_types_spanned"),
    }
    for cid, c in real.items()
])

print("=== Community count ===")
print(f"{len(df):,} communities (size >= {MIN_SIZE})\n")

print("=== Size ===")
print(df["size"].describe(), "\n")

print("=== Persistence ===")
print(df["persistence"].describe(), "\n")

print("=== Cell types spanned ===")
print(df["n_cell_types_spanned"].describe(), "\n")
print("Distribution of n_cell_types_spanned:")
print(df["n_cell_types_spanned"].value_counts().sort_index(), "\n")

# ---- 2. most common genes across all communities ----
# members are "<cell_type>::<gene>" strings; strip to just the gene part
gene_counter = Counter()
for c in real.values():
    for m in c["members"]:
        gene = m.split("::")[-1]
        gene_counter[gene] += 1

top_genes = pd.DataFrame(gene_counter.most_common(30), columns=["gene", "n_communities"])
print("=== Top 30 genes by number of communities they appear in ===")
print(top_genes.to_string(index=False))

# ---- optional: save the per-community table and gene counts to CSV ----
df.to_csv("hidef_community_overview.csv", index=False)
pd.DataFrame(gene_counter.most_common(), columns=["gene", "n_communities"]).to_csv(
    "hidef_gene_frequency.csv", index=False
)
print("\n✓ Saved hidef_community_overview.csv and hidef_gene_frequency.csv")

Total entries in pickle: 143,017
Entries with size >= 2: 2,878

=== Community count ===
2,878 communities (size >= 2)

=== Size ===
count    2878.000000
mean      117.537526
std       429.484467
min        10.000000
25%        12.000000
50%        15.000000
75%        19.000000
max      4678.000000
Name: size, dtype: float64 

=== Persistence ===
count       0
unique      0
top       NaN
freq      NaN
Name: persistence, dtype: object 

=== Cell types spanned ===
count    2878.000000
mean       12.744962
std         5.700764
min         1.000000
25%        11.000000
50%        13.000000
75%        16.000000
max        23.000000
Name: n_cell_types_spanned, dtype: float64 

Distribution of n_cell_types_spanned:
n_cell_types_spanned
1     391
6       1
7       1
8       3
9       3
10    316
11    275
12    227
13    257
14    254
15    217
16    220
17    203
18    150
19    109
20     77
21     43
22     34
23     97
Name: count, dtype: int64 

=== Top 30 genes by number of communities t

In [3]:
import pandas as pd
df = pd.read_csv("11_hidef_results_faithful_cpm/characterized/heterogeneous_hidef_faithful_cpm_cluster_summary.csv")
total_members = df["size"].sum()
total_malformed = df["n_members_malformed"].sum()
total_missing = df["n_members_missing_from_graph"].sum()
print(f"total members across all clusters: {total_members:,}")
print(f"malformed (no '::' or unresolved index): {total_malformed:,} ({total_malformed/total_members:.1%})")
print(f"missing from current supra-graph:        {total_missing:,} ({total_missing/total_members:.1%})")

total members across all clusters: 285,909
malformed (no '::' or unresolved index): 5 (0.0%)
missing from current supra-graph:        1 (0.0%)


In [6]:
import pandas as pd
df = pd.read_csv('../expression_data/neuropathic_pain_drg2.csv', low_memory=False)
for col in df.columns[1:6]:  # spot-check a few
    bad = df[col].apply(lambda x: isinstance(x, str))
    if bad.any():
        print(col, df.loc[bad, col].unique()[:10])

Sample1 ['P' '59' 'M' '11.15510204' '0.232653061' '955.3469388' '5.701020408' '0'
 '29.77040816' '29.3122449']
Sample2 ['P' '61' 'M' '4.897959184' '0.291836735' '1455.708163' '2.53877551'
 '0.183673469' '25.63979592' '41.24693878']
Sample3 ['P' '55' 'M' '9.755102041' '0' '794.5673469' '9.502040816' '52.77959184'
 '0.386734694' '38.71122449']
Sample4 ['P' '55' 'M' '7.136734694' '0' '772.2346939' '2.393877551' '96.86122449'
 '42.51428571' '65.72653061']
Sample5 ['P' '70' 'M' '0' '1249.7' '28.2' '17.9' '32.15' '9.5' '25.6']


In [8]:
df = pd.read_csv('../expression_data/neuropathic_pain_drg2.csv', low_memory=False)
print(df.iloc[:5, 0])

0    pain state
1           age
2           sex
3          A1BG
4          A1CF
Name: GeneName, dtype: object


In [1]:
import csv
import filecmp
from itertools import zip_longest

def read_csv(path):
    with open(path, newline="", encoding="utf-8-sig") as f:  # strips Excel's BOM
        return list(csv.reader(f))

def compare_csv(path1, path2, ignore_row_order=False, max_diffs=10):
    # Fast path: byte-for-byte identical
    if filecmp.cmp(path1, path2, shallow=False):
        print("Files are identical.")
        return True

    a, b = read_csv(path1), read_csv(path2)
    if ignore_row_order:
        a, b = a[:1] + sorted(a[1:]), b[:1] + sorted(b[1:])  # header stays first

    diffs = [(i, x, y) for i, (x, y) in enumerate(zip_longest(a, b), start=1) if x != y]
    if not diffs:
        print("No differing rows (files differ only in formatting or row order).")
        return True

    print(f"Files differ: {len(diffs)} row(s) don't match.\n")
    for i, x, y in diffs[:max_diffs]:
        print(f"Row {i}:\n  file1: {x}\n  file2: {y}")
    if len(diffs) > max_diffs:
        print(f"...and {len(diffs) - max_diffs} more.")
    return False

compare_csv("00_inputs/hdrg_markers.csv", "../R/convert_hdrg_human_2025/cell_type_markers.csv")
# compare_csv("file1.csv", "file2.csv", ignore_row_order=True)

Files are identical.


True